# M5 — E08: fine-tuning `multilingual-e5-large-instruct` (Kaggle)

**Цель.** Проверить, даёт ли domain fine-tuning bi-encoder прирост на frozen
M0 validation proxy и в дальнейшем — в hybrid union. Модель обучается только
на train query groups; validation query groups не участвуют ни в positives, ни
в mining hard negatives. Основная метрика — macro **Recall@50**.

Включите в Kaggle **Internet** (скачать open-weight модель) и **GPU**. Модель
исполняется локально на GPU: внешние inference API не используются. ClearML
нужен только для live-логирования эксперимента.


## План ноутбука

1. Настроить Kaggle, live ClearML и строгий общий лимит **60 минут**.
2. Восстановить group-disjoint M0 proxy и подготовить E5-тексты.
3. Намайнить category-constrained hard negatives zero-shot E5-моделью.
4. Дообучить E5 на triplets `query / positive / hard-negative`.
5. Оценить zero-shot и fine-tuned модель на одной frozen validation proxy.
6. Сохранить переносимый model checkpoint, tokenizer/config, manifest,
   инструкции для M9 и ZIP в `/kaggle/working`.

Бюджет распределён так: до 20 минут на mining, до 21 минуты на обучение,
1 минута на сохранение checkpoint и 18 минут на финальную оценку/ZIP.
Callback сохраняет последние 19 минут для checkpoint и финальной фазы.


## 1. Установка зависимостей и общий time budget


In [ ]:
# Plain install preserves Kaggle's working compiled stack while resolving
# ordinary dependencies such as ClearML's pathlib2.
!pip install -q "clearml>=1.16" "sentence-transformers>=3.4" "transformers>=4.51" "datasets>=3.2" "accelerate>=1.2"


In [ ]:
from __future__ import annotations

import gc
import importlib.metadata
import json
import os
import random
import shutil
import time
import zipfile
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow
import sklearn
import torch
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
MODEL_ID = "intfloat/multilingual-e5-large-instruct"
MODEL_LABEL = "finetuned_multilingual_e5_large_instruct"
QUERY_INSTRUCTION = "Given a Russian service-search request, retrieve relevant service listings."
MAX_SEQ_LENGTH = 256
DESCRIPTION_CHAR_LIMIT = 1_500
HARD_NEGATIVE_TOP_K = 32
VALIDATION_TOP_K = 200
FINAL_K = 50

# The core budget begins after package installation: 20m mining + up to 21m training
# + 1m checkpoint-save buffer + 18m final re-encoding/export = 60 minutes.
TOTAL_BUDGET_SECONDS = 60 * 60
MAX_MINING_SECONDS = 20 * 60
MAX_TRAIN_SECONDS = 21 * 60
CHECKPOINT_SAVE_BUFFER_SECONDS = 1 * 60
FINALIZATION_RESERVE_SECONDS = 18 * 60
# The callback, not the step cap, is the normal stopping condition.
MAX_TRAIN_STEPS = 900
TRAIN_BATCH_SIZE = 6  # T4-safe with gradient checkpointing and max_length=256.
LEARNING_RATE = 1e-5
WARMUP_RATIO = 0.10
WEIGHT_DECAY = 0.01
MAX_TRAIN_EXAMPLES = 80_000
ENCODE_BATCH_SIZE = 48
QUERY_SCORE_BATCH_SIZE = 96
USE_ALL_VISIBLE_GPUS_FOR_ENCODING = True

OUTPUT_DIR = Path("/kaggle/working/m5_e08__finetuned_multilingual_e5_large_instruct")
MODEL_DIR = OUTPUT_DIR / "finetuned_e5_checkpoint"
SCRATCH_DIR = Path("/kaggle/temp/m5_e08_finetune")
HF_CACHE_DIR = Path("/kaggle/temp/hf-cache")
for directory in (OUTPUT_DIR, SCRATCH_DIR, HF_CACHE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("HF_HOME", str(HF_CACHE_DIR))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.pop("CLEARML_OFFLINE_MODE", None)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

NOTEBOOK_STARTED = time.perf_counter()
NOTEBOOK_DEADLINE = NOTEBOOK_STARTED + TOTAL_BUDGET_SECONDS

def seconds_left() -> float:
    return max(0.0, NOTEBOOK_DEADLINE - time.perf_counter())

def require_time(seconds: float, stage: str) -> None:
    if seconds_left() < seconds:
        raise RuntimeError(
            f"Insufficient time for {stage}: {seconds_left() / 60:.1f} minutes remain. "
            "Stop here, save the Kaggle Version, and rerun in a fresh session."
        )

print({
    "model": MODEL_ID,
    "output_dir": str(OUTPUT_DIR),
    "total_budget_minutes": TOTAL_BUDGET_SECONDS / 60,
    "gpu_count": torch.cuda.device_count(),
    "torch": torch.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "pyarrow": pyarrow.__version__,
})
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator before running E08."


## 2. Kaggle Input, ClearML secrets и frozen validation protocol


In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

def get_required_secret(name: str) -> str:
    try:
        value = secrets.get_secret(name)
    except Exception as exc:
        raise RuntimeError(f"Add Kaggle Secret {name!r} before running this notebook.") from exc
    if not value:
        raise RuntimeError(f"Kaggle Secret {name!r} is empty.")
    return value

def get_optional_secret(name: str) -> None:
    try:
        value = secrets.get_secret(name)
    except Exception:
        return
    if value:
        os.environ[name] = value

os.environ["CLEARML_API_ACCESS_KEY"] = get_required_secret("CLEARML_API_ACCESS_KEY")
os.environ["CLEARML_API_SECRET_KEY"] = get_required_secret("CLEARML_API_SECRET_KEY")
for optional_secret in ("CLEARML_API_HOST", "CLEARML_WEB_HOST", "CLEARML_FILES_HOST"):
    get_optional_secret(optional_secret)

INPUT_ROOT = Path("/kaggle/input")

def find_input_file(filename: str) -> Path:
    matches = sorted(INPUT_ROOT.glob(f"**/{filename}"))
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one attached Kaggle Input named {filename!r}; "
            f"found {[str(path) for path in matches]}"
        )
    return matches[0]

TRAIN_PATH = find_input_file("train.parquet")
BENCHMARK_QUERIES_PATH = find_input_file("benchmark_queries.parquet")
BENCHMARK_ITEMS_PATH = find_input_file("benchmark_items.parquet")
print({
    "train": str(TRAIN_PATH),
    "benchmark_queries": str(BENCHMARK_QUERIES_PATH),
    "benchmark_items": str(BENCHMARK_ITEMS_PATH),
    "clearml_credentials_present": True,
})


In [ ]:
from clearml import Task

clearml_task = Task.init(
    project_name="avito-retrieval",
    task_name="E08__finetune_multilingual_e5_large_instruct__kaggle__s42",
    reuse_last_task_id=False,
    # Metrics and artifacts are logged explicitly below; this avoids slow
    # automatic notebook hooks observed in Kaggle.
    auto_connect_arg_parser=False,
    auto_connect_frameworks={"detect_repository": False},
    auto_resource_monitoring=False,
    auto_connect_streams=False,
)
clearml_task.connect(
    {
        "stage": "M5_E08_finetuned_bi_encoder",
        "environment": "kaggle",
        "model_id": MODEL_ID,
        "validation_protocol": "benchmark_aligned_group_disjoint_proxy_v1",
        "seed": SEED,
        "total_budget_minutes": TOTAL_BUDGET_SECONDS / 60,
        "max_mining_minutes": MAX_MINING_SECONDS / 60,
        "max_training_minutes": MAX_TRAIN_SECONDS / 60,
        "checkpoint_save_buffer_minutes": CHECKPOINT_SAVE_BUFFER_SECONDS / 60,
        "finalization_reserve_minutes": FINALIZATION_RESERVE_SECONDS / 60,
        "hard_negative_top_k": HARD_NEGATIVE_TOP_K,
        "max_train_steps": MAX_TRAIN_STEPS,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "category_rule": "item_category_id == search_category; full-corpus fallback only when category is absent",
    },
    name="config",
)
clearml_logger = clearml_task.get_logger()
print({"clearml_task_id": clearml_task.id, "offline_mode": False, "logging": "manual"})


In [ ]:
SEARCH_COLUMNS = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]
TRAIN_COLUMNS = [*SEARCH_COLUMNS, "item_id"]
BENCHMARK_QUERY_COLUMNS = ["query_id", *SEARCH_COLUMNS]
ITEM_COLUMNS = [
    "item_id",
    "item_title_raw",
    "item_description_raw",
    "item_infm_params_text",
    "item_category_id",
]

def canonical_query_frame(frame: pd.DataFrame) -> pd.DataFrame:
    # Canonical fields only for grouping equivalent request contexts.
    result = frame[SEARCH_COLUMNS].copy()
    for column in ("search_query", "search_infm_params_text"):
        result[column] = (
            result[column]
            .astype("string")
            .fillna("<NA>")
            .str.lower()
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )
    for column in ("search_location_id", "search_is_delivery_search", "search_category"):
        result[column] = result[column].astype("string").fillna("<NA>")
    return result

def compose_query_text(frame: pd.DataFrame) -> list[str]:
    query = frame["search_query"].fillna("").astype(str).str.strip()
    filters = frame["search_infm_params_text"].fillna("").astype(str).str.strip()
    return [
        value if not params else f"{value}\nФильтры поиска: {params}"
        for value, params in zip(query, filters, strict=True)
    ]

def compose_item_text(frame: pd.DataFrame) -> list[str]:
    title = frame["item_title_raw"].fillna("").astype(str).str.strip()
    params = frame["item_infm_params_text"].fillna("").astype(str).str.strip()
    description = (
        frame["item_description_raw"].fillna("").astype(str)
        .str.slice(stop=DESCRIPTION_CHAR_LIMIT).str.strip()
    )
    texts: list[str] = []
    for title_value, params_value, description_value in zip(title, params, description, strict=True):
        parts = [f"Название услуги: {title_value}"]
        if params_value:
            parts.append(f"Параметры: {params_value}")
        if description_value:
            parts.append(f"Описание: {description_value}")
        texts.append("\n".join(parts))
    return texts

def e5_query_texts(texts: list[str]) -> list[str]:
    return [f"Instruct: {QUERY_INSTRUCTION}\nQuery: {text}" for text in texts]

def load_frozen_proxy() -> dict[str, object]:
    train_pairs = pd.read_parquet(TRAIN_PATH, columns=TRAIN_COLUMNS)
    benchmark_queries = pd.read_parquet(BENCHMARK_QUERIES_PATH, columns=BENCHMARK_QUERY_COLUMNS)
    candidate_items = pd.read_parquet(BENCHMARK_ITEMS_PATH, columns=ITEM_COLUMNS).reset_index(drop=True)
    candidate_items["item_id"] = candidate_items["item_id"].astype(str)
    train_pairs["item_id"] = train_pairs["item_id"].astype(str)

    all_contexts = pd.concat(
        [canonical_query_frame(train_pairs), canonical_query_frame(benchmark_queries)],
        ignore_index=True,
    )
    group_ids, _ = pd.factorize(pd.MultiIndex.from_frame(all_contexts), sort=False)
    train_pairs["query_group"] = group_ids[: len(train_pairs)]

    candidate_item_ids = candidate_items["item_id"].to_numpy()
    candidate_item_id_set = set(candidate_item_ids)
    proxy_pairs = train_pairs.loc[train_pairs["item_id"].isin(candidate_item_id_set)].copy()
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
    train_idx, valid_idx = next(splitter.split(proxy_pairs, groups=proxy_pairs["query_group"]))
    proxy_train_pairs = proxy_pairs.iloc[train_idx].copy()
    proxy_valid_pairs = proxy_pairs.iloc[valid_idx].copy()
    assert set(proxy_train_pairs["query_group"]).isdisjoint(set(proxy_valid_pairs["query_group"]))

    validation_queries = (
        proxy_valid_pairs.sort_values("query_group")
        .drop_duplicates("query_group")
        .loc[:, ["query_group", *SEARCH_COLUMNS]]
        .reset_index(drop=True)
    )
    gold_by_group = (
        proxy_valid_pairs.groupby("query_group", sort=False)["item_id"]
        .agg(lambda values: frozenset(values.astype(str)))
        .to_dict()
    )
    gold_sets = [gold_by_group[group] for group in validation_queries["query_group"]]

    category_to_indices = {
        str(category): group.index.to_numpy(dtype=np.int64)
        for category, group in candidate_items.groupby("item_category_id", sort=False)
    }
    all_indices = np.arange(len(candidate_items), dtype=np.int64)
    allowed_indices_by_query = [
        category_to_indices.get(str(category), all_indices)
        for category in validation_queries["search_category"]
    ]
    category_oracle = float(np.mean([
        len(gold & set(candidate_item_ids[allowed])) / len(gold)
        for gold, allowed in zip(gold_sets, allowed_indices_by_query, strict=True)
    ]))
    assert category_oracle == 1.0, "Category partition removed a validation positive."

    return {
        "proxy_train_pairs": proxy_train_pairs,
        "proxy_valid_pairs": proxy_valid_pairs,
        "validation_queries": validation_queries,
        "gold_sets": gold_sets,
        "candidate_items": candidate_items,
        "candidate_item_ids": candidate_item_ids,
        "category_to_indices": category_to_indices,
        "all_indices": all_indices,
        "category_oracle": category_oracle,
        "benchmark_queries": benchmark_queries,
    }

load_started = time.perf_counter()
proxy = load_frozen_proxy()
candidate_items = proxy["candidate_items"]
candidate_item_ids = proxy["candidate_item_ids"]
candidate_item_texts = compose_item_text(candidate_items)
validation_queries = proxy["validation_queries"]
validation_query_texts = compose_query_text(validation_queries)
print({
    "load_seconds": round(time.perf_counter() - load_started, 2),
    "candidate_items": len(candidate_items),
    "proxy_train_positive_rows": len(proxy["proxy_train_pairs"]),
    "proxy_valid_positive_rows": len(proxy["proxy_valid_pairs"]),
    "validation_query_groups": len(validation_queries),
    "category_oracle_recall_at_50": proxy["category_oracle"],
    "minutes_left": round(seconds_left() / 60, 1),
})


## 3. Hard-negative mining

Mining использует исходный E5 только как локальный teacher: для каждого train
query group берётся первый non-positive из top-32 dense кандидатов **в той же
категории**. Позитивы и known positives от validation groups не используются.


In [ ]:
from sentence_transformers import SentenceTransformer

def hardware_snapshot() -> dict[str, object]:
    return {
        "cuda_available": bool(torch.cuda.is_available()),
        "gpu_count": int(torch.cuda.device_count()),
        "gpus": [torch.cuda.get_device_name(index) for index in range(torch.cuda.device_count())],
    }

def load_encoder_for_encoding(model_name_or_path: str, *, use_pool: bool) -> tuple[SentenceTransformer, list[str]]:
    devices = [f"cuda:{index}" for index in range(torch.cuda.device_count())]
    if not devices:
        raise RuntimeError("GPU is required for E08.")
    encoding_devices = devices if use_pool and len(devices) > 1 else devices[:1]
    # Parent stays on CPU when start_multi_process_pool owns two GPU workers.
    model = SentenceTransformer(model_name_or_path, device="cpu" if len(encoding_devices) > 1 else encoding_devices[0])
    model.max_seq_length = MAX_SEQ_LENGTH
    if len(encoding_devices) == 1:
        model.half()
    return model, encoding_devices

def encode_e5(
    model: SentenceTransformer,
    texts: list[str],
    *,
    is_query: bool,
    pool: object | None,
) -> np.ndarray:
    prepared = e5_query_texts(texts) if is_query else texts
    kwargs: dict[str, object] = {
        "batch_size": ENCODE_BATCH_SIZE,
        "show_progress_bar": True,
        "convert_to_numpy": True,
        "normalize_embeddings": True,
    }
    if pool is not None:
        kwargs.update({"pool": pool, "chunk_size": 1_000})
    vectors = model.encode(prepared, **kwargs)
    return np.asarray(vectors, dtype=np.float32)

def exact_category_topk(
    query_embeddings: np.ndarray,
    query_categories: list[object],
    item_embeddings: np.ndarray,
    *,
    top_k: int,
) -> tuple[list[list[str]], int]:
    # Exact inner-product search inside a category, batched by category.
    rankings: list[list[str] | None] = [None] * len(query_embeddings)
    query_positions_by_category: dict[str, list[int]] = defaultdict(list)
    for position, category in enumerate(query_categories):
        query_positions_by_category[str(category)].append(position)

    fallback_queries = 0
    for category, positions in query_positions_by_category.items():
        allowed = proxy["category_to_indices"].get(category)
        if allowed is None:
            allowed = proxy["all_indices"]
            fallback_queries += len(positions)
        k = min(top_k, len(allowed))
        for start in range(0, len(positions), QUERY_SCORE_BATCH_SIZE):
            current_positions = positions[start : start + QUERY_SCORE_BATCH_SIZE]
            scores = query_embeddings[current_positions] @ item_embeddings[allowed].T
            partition = np.argpartition(-scores, kth=k - 1, axis=1)[:, :k]
            partition_scores = np.take_along_axis(scores, partition, axis=1)
            order = np.argsort(-partition_scores, axis=1)
            ranked_item_indices = allowed[np.take_along_axis(partition, order, axis=1)]
            for position, row in zip(current_positions, ranked_item_indices, strict=True):
                rankings[position] = candidate_item_ids[row].astype(str).tolist()
    assert all(row is not None for row in rankings)
    return [row for row in rankings if row is not None], fallback_queries

def evaluate_rankings(rankings: list[list[str]], gold_sets: list[frozenset[str]]) -> dict[str, float]:
    result: dict[str, float] = {}
    for k in (1, 5, 10, 20, 50, 200):
        result[f"recall@{k}"] = float(np.mean([
            len(set(ranking[:k]) & gold) / len(gold)
            for ranking, gold in zip(rankings, gold_sets, strict=True)
        ]))
    result["hit_rate@50"] = float(np.mean([
        bool(set(ranking[:50]) & gold)
        for ranking, gold in zip(rankings, gold_sets, strict=True)
    ]))
    return result

def write_validation_rankings(path: Path, rankings: list[list[str]], *, model_name: str) -> None:
    with path.open("w", encoding="utf-8") as handle:
        for query_group, ranking in zip(validation_queries["query_group"], rankings, strict=True):
            handle.write(json.dumps({
                "query_group": int(query_group),
                "model": model_name,
                "item_ids": ranking,
            }, ensure_ascii=False) + "\n")

train_query_frame = (
    proxy["proxy_train_pairs"].sort_values("query_group")
    .drop_duplicates("query_group")
    .loc[:, ["query_group", *SEARCH_COLUMNS]]
    .reset_index(drop=True)
)
train_query_texts = compose_query_text(train_query_frame)
print({
    "train_query_groups_for_mining": len(train_query_frame),
    "hard_negative_top_k": HARD_NEGATIVE_TOP_K,
    "hardware": hardware_snapshot(),
})


In [ ]:
require_time(FINALIZATION_RESERVE_SECONDS + MAX_MINING_SECONDS, "zero-shot mining")
mining_started = time.perf_counter()
mining_deadline = min(NOTEBOOK_DEADLINE - FINALIZATION_RESERVE_SECONDS, mining_started + MAX_MINING_SECONDS)

base_model, encoding_devices = load_encoder_for_encoding(
    MODEL_ID,
    use_pool=USE_ALL_VISIBLE_GPUS_FOR_ENCODING,
)
pool = None
try:
    if len(encoding_devices) > 1:
        pool = base_model.start_multi_process_pool(target_devices=encoding_devices)

    item_embeddings_base = encode_e5(base_model, candidate_item_texts, is_query=False, pool=pool)
    train_query_embeddings_base = encode_e5(base_model, train_query_texts, is_query=True, pool=pool)
    validation_query_embeddings_base = encode_e5(base_model, validation_query_texts, is_query=True, pool=pool)

    train_dense_rankings, train_fallback_count = exact_category_topk(
        train_query_embeddings_base,
        train_query_frame["search_category"].tolist(),
        item_embeddings_base,
        top_k=HARD_NEGATIVE_TOP_K,
    )
    baseline_rankings, validation_fallback_count = exact_category_topk(
        validation_query_embeddings_base,
        validation_queries["search_category"].tolist(),
        item_embeddings_base,
        top_k=VALIDATION_TOP_K,
    )
finally:
    if pool is not None:
        base_model.stop_multi_process_pool(pool)
    del base_model
    torch.cuda.empty_cache()
    gc.collect()

baseline_metrics = evaluate_rankings(baseline_rankings, proxy["gold_sets"])
write_validation_rankings(
    OUTPUT_DIR / "validation_top200__zero_shot_e5.jsonl",
    baseline_rankings,
    model_name="zero_shot_multilingual_e5_large_instruct",
)
print({
    "mining_seconds": round(time.perf_counter() - mining_started, 2),
    "mining_budget_seconds": MAX_MINING_SECONDS,
    "baseline_recall_at_50": baseline_metrics["recall@50"],
    "train_category_fallback_queries": train_fallback_count,
    "validation_category_fallback_queries": validation_fallback_count,
    "minutes_left": round(seconds_left() / 60, 1),
})
if time.perf_counter() > mining_deadline:
    print("Warning: mining exceeded its soft budget; training callback will preserve finalization reserve.")


## 4. Fine-tuning на mined triplets

Каждая строка — `anchor=query`, `positive=chosen item`,
`negative=category-constrained dense hard negative`. `MultipleNegativesRankingLoss`
использует также in-batch negatives. Обучение ограничено и max steps, и
wall-clock callback; сохраняется последний переносимый checkpoint.


In [ ]:
known_positive_by_group = (
    proxy["proxy_train_pairs"].groupby("query_group", sort=False)["item_id"]
    .agg(lambda values: frozenset(values.astype(str)))
    .to_dict()
)
train_category_by_group = dict(zip(
    train_query_frame["query_group"].astype(int),
    train_query_frame["search_category"].astype(str),
    strict=True,
))
hard_negative_by_group: dict[int, str] = {}
random_fallback_groups = 0

for query_group, ranking in zip(train_query_frame["query_group"], train_dense_rankings, strict=True):
    group_id = int(query_group)
    positives = known_positive_by_group[group_id]
    negative = next((item_id for item_id in ranking if item_id not in positives), None)
    if negative is None:
        # This rare fallback remains category-constrained whenever possible.
        allowed = proxy["category_to_indices"].get(train_category_by_group[group_id], proxy["all_indices"])
        for index in allowed:
            candidate_id = str(candidate_item_ids[index])
            if candidate_id not in positives:
                negative = candidate_id
                random_fallback_groups += 1
                break
    if negative is not None:
        hard_negative_by_group[group_id] = negative

train_query_text_by_group = dict(zip(
    train_query_frame["query_group"].astype(int),
    train_query_texts,
    strict=True,
))
item_text_by_id = dict(zip(candidate_item_ids, candidate_item_texts, strict=True))
unique_train_pairs = proxy["proxy_train_pairs"][["query_group", "item_id"]].drop_duplicates().copy()
unique_train_pairs["query_group"] = unique_train_pairs["query_group"].astype(int)
unique_train_pairs["item_id"] = unique_train_pairs["item_id"].astype(str)
unique_train_pairs["negative_item_id"] = unique_train_pairs["query_group"].map(hard_negative_by_group)
unique_train_pairs = unique_train_pairs.dropna(subset=["negative_item_id"])
unique_train_pairs = unique_train_pairs.loc[
    unique_train_pairs["item_id"] != unique_train_pairs["negative_item_id"]
].reset_index(drop=True)

if len(unique_train_pairs) > MAX_TRAIN_EXAMPLES:
    unique_train_pairs = unique_train_pairs.sample(MAX_TRAIN_EXAMPLES, random_state=SEED).reset_index(drop=True)

train_examples = pd.DataFrame({
    "anchor": [e5_query_texts([train_query_text_by_group[group]])[0] for group in unique_train_pairs["query_group"]],
    "positive": [item_text_by_id[item_id] for item_id in unique_train_pairs["item_id"]],
    "negative": [item_text_by_id[item_id] for item_id in unique_train_pairs["negative_item_id"]],
})
assert not train_examples.empty, "No valid fine-tuning triplets were constructed."

mined_pairs_path = OUTPUT_DIR / "mined_train_triplets.parquet"
unique_train_pairs.to_parquet(mined_pairs_path, index=False)
print({
    "unique_train_pairs": len(unique_train_pairs),
    "train_query_groups": unique_train_pairs["query_group"].nunique(),
    "groups_with_hard_negative": len(hard_negative_by_group),
    "fallback_negative_groups": random_fallback_groups,
    "effective_train_examples": len(train_examples),
})


In [ ]:
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
)
from sentence_transformers.training_args import BatchSamplers
from transformers import TrainerCallback

class StopBeforeDeadlineCallback(TrainerCallback):
    # Stop at step boundaries so final encoding/export gets its reserve.

    def __init__(self, deadline: float) -> None:
        self.deadline = deadline
        self.triggered = False

    def on_step_end(self, args, state, control, **kwargs):
        if time.perf_counter() >= self.deadline:
            if not self.triggered:
                print("Training time budget reached; stopping after this optimizer step.")
                self.triggered = True
            control.should_training_stop = True
        return control

require_time(FINALIZATION_RESERVE_SECONDS + 5 * 60, "fine-tuning")
training_started = time.perf_counter()
training_deadline = min(
    NOTEBOOK_DEADLINE - FINALIZATION_RESERVE_SECONDS - CHECKPOINT_SAVE_BUFFER_SECONDS,
    training_started + MAX_TRAIN_SECONDS,
)

training_model = SentenceTransformer(MODEL_ID, device="cuda:0")
training_model.max_seq_length = MAX_SEQ_LENGTH
first_module = training_model._first_module()
auto_model = getattr(first_module, "auto_model", None)
if auto_model is not None and hasattr(auto_model, "gradient_checkpointing_enable"):
    auto_model.gradient_checkpointing_enable()
    if hasattr(auto_model.config, "use_cache"):
        auto_model.config.use_cache = False

train_dataset = Dataset.from_pandas(train_examples, preserve_index=False)
trainer_checkpoint_dir = SCRATCH_DIR / "trainer_checkpoints"
training_args = SentenceTransformerTrainingArguments(
    output_dir=str(trainer_checkpoint_dir),
    num_train_epochs=999,
    max_steps=MAX_TRAIN_STEPS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    logging_strategy="steps",
    logging_steps=25,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=1,
    report_to=[],
    seed=SEED,
    data_seed=SEED,
    dataloader_num_workers=2,
    remove_unused_columns=False,
    # A duplicate query or listing inside one batch would become a false
    # in-batch negative for MultipleNegativesRankingLoss.
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    run_name="E08_finetuned_multilingual_e5_large_instruct",
)
trainer = SentenceTransformerTrainer(
    model=training_model,
    args=training_args,
    train_dataset=train_dataset,
    loss=losses.MultipleNegativesRankingLoss(training_model),
    callbacks=[StopBeforeDeadlineCallback(training_deadline)],
)
train_result = trainer.train()

MODEL_DIR.mkdir(parents=True, exist_ok=True)
training_model.save(str(MODEL_DIR))
trainer_state_path = OUTPUT_DIR / "trainer_state.json"
trainer.state.save_to_json(str(trainer_state_path))
training_config = {
    "base_model": MODEL_ID,
    "query_instruction": QUERY_INSTRUCTION,
    "max_seq_length": MAX_SEQ_LENGTH,
    "loss": "MultipleNegativesRankingLoss(anchor, positive, hard_negative)",
    "hard_negative_source": "zero-shot E5 top-32, category-constrained, positives excluded",
    "group_split": "GroupShuffleSplit(seed=42, test_size=0.20)",
    "max_train_steps": MAX_TRAIN_STEPS,
    "actual_global_step": int(trainer.state.global_step),
    "train_batch_size": TRAIN_BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": WARMUP_RATIO,
    "weight_decay": WEIGHT_DECAY,
    "training_seconds": time.perf_counter() - training_started,
    "time_left_seconds_after_training": seconds_left(),
}
(OUTPUT_DIR / "training_config.json").write_text(
    json.dumps(training_config, ensure_ascii=False, indent=2), encoding="utf-8"
)
print({
    "global_step": int(trainer.state.global_step),
    "training_seconds": round(training_config["training_seconds"], 2),
    "minutes_left": round(seconds_left() / 60, 1),
    "checkpoint": str(MODEL_DIR),
})

del trainer, train_dataset, training_model, train_examples
del item_embeddings_base, train_query_embeddings_base, validation_query_embeddings_base
torch.cuda.empty_cache()
gc.collect()


## 5. Group-aware validation после обучения

Для честного сравнения кодируем benchmark corpus заново fine-tuned моделью и
считаем exact category retrieval на тех же held-out query groups. Использовать
pre-training item embeddings после fine-tuning запрещено.


In [ ]:
post_eval_started = time.perf_counter()
post_metrics: dict[str, object]
finetuned_rankings: list[list[str]] = []

# Starting re-encoding without its full reserve risks losing the final ZIP.
if seconds_left() >= FINALIZATION_RESERVE_SECONDS:
    final_model, final_encoding_devices = load_encoder_for_encoding(
        str(MODEL_DIR),
        use_pool=USE_ALL_VISIBLE_GPUS_FOR_ENCODING,
    )
    final_pool = None
    try:
        if len(final_encoding_devices) > 1:
            final_pool = final_model.start_multi_process_pool(target_devices=final_encoding_devices)
        item_embeddings_finetuned = encode_e5(final_model, candidate_item_texts, is_query=False, pool=final_pool)
        validation_query_embeddings_finetuned = encode_e5(
            final_model, validation_query_texts, is_query=True, pool=final_pool
        )
        finetuned_rankings, finetuned_fallback_count = exact_category_topk(
            validation_query_embeddings_finetuned,
            validation_queries["search_category"].tolist(),
            item_embeddings_finetuned,
            top_k=VALIDATION_TOP_K,
        )
    finally:
        if final_pool is not None:
            final_model.stop_multi_process_pool(final_pool)
        del final_model
        torch.cuda.empty_cache()
        gc.collect()

    post_metrics = evaluate_rankings(finetuned_rankings, proxy["gold_sets"])
    post_metrics["category_fallback_queries"] = int(finetuned_fallback_count)
    post_metrics["status"] = "completed"
    write_validation_rankings(
        OUTPUT_DIR / "validation_top200__finetuned_e5.jsonl",
        finetuned_rankings,
        model_name=MODEL_LABEL,
    )
else:
    post_metrics = {
        "status": "skipped_insufficient_time",
        "reason": "The model checkpoint was saved; rerun a fresh Kaggle session to evaluate it.",
    }

print({
    "post_eval_seconds": round(time.perf_counter() - post_eval_started, 2),
    "post_status": post_metrics["status"],
    "finetuned_recall_at_50": post_metrics.get("recall@50"),
    "zero_shot_recall_at_50": baseline_metrics["recall@50"],
    "minutes_left": round(seconds_left() / 60, 1),
})


## 6. Артефакты для M9, ClearML и ZIP


In [ ]:
metrics_rows = [
    {"model": "zero_shot_multilingual_e5_large_instruct", "stage": "before_finetune", **baseline_metrics},
    {"model": MODEL_LABEL, "stage": "after_finetune", **post_metrics},
]
metrics_frame = pd.DataFrame(metrics_rows)
metrics_path = OUTPUT_DIR / "validation_metrics.csv"
metrics_frame.to_csv(metrics_path, index=False)

reuse_instructions = f'''# Reuse E08 model in M9

This directory is a local SentenceTransformer checkpoint fine-tuned from `{MODEL_ID}`.

1. Save this notebook as a Kaggle Version. Its `/kaggle/working` output can then be
   attached to the M9 Kaggle notebook as a Kaggle Dataset. If only the ZIP is
   available, extract it first.
2. In M9 set `FINETUNED_E5_PATH` to the attached
   `m5_e08__finetuned_multilingual_e5_large_instruct/finetuned_e5_checkpoint` directory.
3. Load locally with `SentenceTransformer(FINETUNED_E5_PATH)`; set
   `model.max_seq_length = {MAX_SEQ_LENGTH}`.
4. Reuse exactly this text format:
   - query: `Instruct: {QUERY_INSTRUCTION}\nQuery: <search_query plus optional filters>`;
   - item: title + optional infm params + first {DESCRIPTION_CHAR_LIMIT} description characters.
5. Re-encode **all benchmark_items** in M9. Validation embeddings stored by this
   notebook are only diagnostic and must never be used for the final benchmark.
6. First compare standalone Recall@50 and the retained union quota on the frozen proxy.
   Current zero-shot dense quota is 10 candidates; keep the fine-tuned model only if
   its M1+M2+dense union improves over the frozen baseline.
'''
reuse_path = OUTPUT_DIR / "M9_REUSE_INSTRUCTIONS.md"
reuse_path.write_text(reuse_instructions, encoding="utf-8")

model_files = [
    {
        "path": path.relative_to(MODEL_DIR).as_posix(),
        "bytes": path.stat().st_size,
    }
    for path in sorted(MODEL_DIR.rglob("*"))
    if path.is_file()
]
assert (MODEL_DIR / "modules.json").exists(), "SentenceTransformer checkpoint is incomplete."
assert any(path["path"].endswith((".safetensors", ".bin")) for path in model_files), "No model weight file saved."

manifest = {
    "stage": "M5_E08_finetuned_bi_encoder",
    "base_model": MODEL_ID,
    "export_model_dir": str(MODEL_DIR),
    "model_format": "SentenceTransformer local checkpoint",
    "validation_protocol": "benchmark_aligned_group_disjoint_proxy_v1",
    "seed": SEED,
    "hardware": hardware_snapshot(),
    "time_budget": {
        "total_seconds": TOTAL_BUDGET_SECONDS,
        "elapsed_seconds": time.perf_counter() - NOTEBOOK_STARTED,
        "seconds_left": seconds_left(),
    },
    "training": training_config,
    "zero_shot_metrics": baseline_metrics,
    "finetuned_metrics": post_metrics,
    "artifacts": {
        "model_checkpoint": MODEL_DIR.name,
        "tokenizer_and_model_config": "inside finetuned_e5_checkpoint/0_Transformer",
        "training_state": trainer_state_path.name,
        "training_config": "training_config.json",
        "mined_triplets": mined_pairs_path.name,
        "m9_reuse_instructions": reuse_path.name,
        "validation_metrics": metrics_path.name,
    },
    "model_files": model_files,
    "package_versions": {
        package: importlib.metadata.version(package)
        for package in ("clearml", "sentence-transformers", "transformers", "datasets", "accelerate", "torch")
    },
}
manifest_path = OUTPUT_DIR / "run_manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

# Do not upload multi-GB weights to ClearML during the deadline. Kaggle Version
# is the model transport; ClearML receives lightweight, inspectable evidence.
clearml_logger.report_table("E08 validation", "metrics", 0, table_plot=metrics_frame)
clearml_logger.report_scalar("Recall", "zero_shot@50", baseline_metrics["recall@50"], 0)
if "recall@50" in post_metrics:
    clearml_logger.report_scalar("Recall", "finetuned@50", float(post_metrics["recall@50"]), 0)
    clearml_task.set_parameter("results/finetuned_recall_at_50", float(post_metrics["recall@50"]))
clearml_task.set_parameter("results/zero_shot_recall_at_50", float(baseline_metrics["recall@50"]))
clearml_task.set_parameter("artifacts/model_checkpoint", str(MODEL_DIR))
for artifact_name, artifact_path in {
    "e08_metrics": metrics_path,
    "e08_manifest": manifest_path,
    "e08_m9_reuse": reuse_path,
    "e08_training_config": OUTPUT_DIR / "training_config.json",
}.items():
    clearml_task.upload_artifact(artifact_name, artifact_object=artifact_path)

print({
    "checkpoint": str(MODEL_DIR),
    "checkpoint_files": len(model_files),
    "metrics": str(metrics_path),
    "manifest": str(manifest_path),
    "m9_reuse_instructions": str(reuse_path),
})


In [ ]:
from IPython.display import FileLink, display

zip_path = Path("/kaggle/working/m5_e08__finetuned_multilingual_e5_large_instruct.zip")
required_paths = [
    MODEL_DIR / "modules.json",
    OUTPUT_DIR / "training_config.json",
    OUTPUT_DIR / "trainer_state.json",
    OUTPUT_DIR / "run_manifest.json",
    OUTPUT_DIR / "M9_REUSE_INSTRUCTIONS.md",
    OUTPUT_DIR / "validation_metrics.csv",
]
missing = [str(path) for path in required_paths if not path.exists()]
assert not missing, f"Missing required E08 artifacts: {missing}"

# model.safetensors is already compressed. ZIP_STORED is materially faster and
# helps the notebook stay within the runtime budget.
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
    for path in sorted(OUTPUT_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=path.relative_to(OUTPUT_DIR.parent).as_posix())

clearml_task.close()
print({
    "zip": str(zip_path),
    "zip_gb": round(zip_path.stat().st_size / 1024**3, 3),
    "elapsed_minutes": round((time.perf_counter() - NOTEBOOK_STARTED) / 60, 2),
    "minutes_left": round(seconds_left() / 60, 2),
})
display(FileLink(zip_path))
